# 3D Gaussian Splatting：实时三维场景渲染

这个 Notebook 从零解析 `3D Gaussian Splatting（3DGS）` 的核心原理，并用 NumPy/PyTorch 实现 2D 版本演示。

内容包括：
- 3D 高斯椭球的数学表达
- Alpha 合成（Splatting）渲染流程
- 2D Gaussian Splatting 从零实现与训练
- 高斯参数优化（位置、协方差、颜色、不透明度）
- 与 NeRF 的全面对比
- 密度自适应控制（Adaptive Density Control）原理

## 1. 环境准备

```bash
pip install torch torchvision matplotlib numpy
```

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 图像分辨率
    H: int = 64
    W: int = 64
    # 初始高斯点数量
    n_gaussians: int = 200
    lr_pos: float = 1e-3
    lr_color: float = 1e-2
    lr_opacity: float = 1e-2
    lr_scale: float = 1e-3
    n_iters: int = 3000

cfg = Config()
cfg

## 2. 3D Gaussian Splatting 核心原理

### 2.1 场景表达

3DGS 用一组 **三维高斯椭球**显式表示场景。每个高斯 $G_k$ 由以下参数描述：

| 参数 | 维度 | 含义 |
|------|------|------|
| $\boldsymbol{\mu}_k$ | 3 | 中心位置（x, y, z） |
| $\boldsymbol{\Sigma}_k$ | 3×3 | 协方差矩阵（控制形状和朝向） |
| $\mathbf{c}_k$ | 3 (RGB) 或 SH | 颜色（球谐函数，与观察方向相关） |
| $\alpha_k$ | 1 | 不透明度 |

### 2.2 协方差矩阵的参数化

直接优化 $\Sigma$ 难以保证半正定性，3DGS 改用缩放向量 $\mathbf{s}$ 和旋转四元数 $\mathbf{q}$ 分解：

$$\boldsymbol{\Sigma} = RSS^T R^T$$

其中 $S = \text{diag}(s_1, s_2, s_3)$，$R$ 由四元数 $\mathbf{q}$ 构造。

### 2.3 渲染（Splatting）

1. **投影**：将 3D 高斯投影到 2D 图像平面，得到 2D 高斯
2. **排序**：按深度从后往前排序（或从前往后做 alpha 合成）
3. **Alpha 合成**：

$$C = \sum_{k=1}^{N} \mathbf{c}_k \alpha_k \prod_{j=1}^{k-1}(1 - \alpha_j G_j(\mathbf{x}))$$

### 2.4 与 NeRF 的本质差异

- **NeRF**：隐式表达（MLP），查询慢，渲染需大量 MLP 前向传播
- **3DGS**：显式表达（高斯点云），光栅化渲染，**可实时**（>100 FPS）

In [ ]:
# 可视化单个 2D 高斯的形状
def draw_gaussian_2d(mu, sigma, ax, color='blue', alpha=0.5, n_std=2):
    from matplotlib.patches import Ellipse
    import numpy.linalg as la

    vals, vecs = la.eigh(sigma)
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mu, width=w, height=h, angle=angle,
                  facecolor=color, alpha=alpha, edgecolor=color, linewidth=2)
    ax.add_patch(ell)


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
examples = [
    ((0, 0), np.array([[1.0, 0.0], [0.0, 1.0]]),   '各向同性（圆形）'),
    ((0, 0), np.array([[2.0, 0.0], [0.0, 0.5]]),   '各向异性（椭圆）'),
    ((0, 0), np.array([[1.0, 0.8], [0.8, 1.0]]),   '旋转椭圆'),
]

for ax, (mu, sigma, title) in zip(axes, examples):
    draw_gaussian_2d(mu, sigma, ax)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.grid(True)

plt.suptitle('3D 高斯的 2D 截面形状示意')
plt.tight_layout()
plt.show()

## 3. 2D Gaussian Splatting 实现

In [ ]:
class GaussianSplatting2D(nn.Module):
    def __init__(self, n_gaussians, H, W):
        super().__init__()
        # 中心位置：归一化到 [-1, 1]
        self.positions = nn.Parameter(torch.rand(n_gaussians, 2) * 2 - 1)
        # 缩放参数（对数域，保证正数）
        self.log_scales = nn.Parameter(torch.full((n_gaussians, 2), -2.5))
        # 旋转角（2D 只需一个角度）
        self.rotations  = nn.Parameter(torch.zeros(n_gaussians))
        # 颜色（Sigmoid 后 ∈ [0,1]）
        self.colors     = nn.Parameter(torch.rand(n_gaussians, 3))
        # 不透明度（Sigmoid 后 ∈ [0,1]）
        self.opacities  = nn.Parameter(torch.zeros(n_gaussians))
        self.H = H
        self.W = W

    def forward(self):
        H, W = self.H, self.W
        # 像素坐标网格，归一化到 [-1, 1]
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, H, device=self.positions.device),
            torch.linspace(-1, 1, W, device=self.positions.device),
            indexing='ij'
        )
        grid = torch.stack([xx, yy], dim=-1)  # H, W, 2

        scales = torch.exp(self.log_scales).clamp(1e-4, 0.5)  # N, 2
        cos_r  = torch.cos(self.rotations)  # N
        sin_r  = torch.sin(self.rotations)  # N

        # 构建旋转矩阵 R (N, 2, 2)
        R = torch.stack([cos_r, -sin_r, sin_r, cos_r], dim=-1).reshape(-1, 2, 2)
        # 构建协方差 Σ = R S S^T R^T
        S  = torch.diag_embed(scales)  # N, 2, 2
        RS = R @ S
        cov = RS @ RS.transpose(-1, -2)  # N, 2, 2
        cov_inv = torch.inverse(cov + 1e-6 * torch.eye(2, device=cov.device).unsqueeze(0))

        # 计算每个像素到每个高斯的马氏距离
        diff = grid.unsqueeze(2) - self.positions.unsqueeze(0).unsqueeze(0)  # H, W, N, 2
        mahal = (diff.unsqueeze(-2) @ cov_inv.unsqueeze(0).unsqueeze(0) @ diff.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        # 高斯权重
        gauss = torch.exp(-0.5 * mahal)  # H, W, N

        opacities = torch.sigmoid(self.opacities)  # N
        alpha = gauss * opacities.unsqueeze(0).unsqueeze(0)  # H, W, N

        colors = torch.sigmoid(self.colors)  # N, 3

        # 前到后 alpha 合成（按 x 坐标近似排序）
        depth_order = self.positions[:, 0].argsort()
        alpha  = alpha[..., depth_order]
        colors = colors[depth_order]

        T = torch.cumprod(torch.cat([torch.ones(H, W, 1, device=alpha.device), 1 - alpha[..., :-1] + 1e-8], dim=-1), dim=-1)
        weights = T * alpha  # H, W, N

        img = (weights.unsqueeze(-1) * colors.unsqueeze(0).unsqueeze(0)).sum(dim=2)  # H, W, 3
        # 背景（白色）
        bg = (1 - weights.sum(dim=-1, keepdim=True)).clamp(0, 1)
        img = img + bg

        return img.clamp(0, 1)


print('2D Gaussian Splatting 模块定义完成')

## 4. 目标图像生成

In [ ]:
def make_target_image(H, W):
    # 合成目标：几个彩色圆盘
    img = np.ones((H, W, 3), dtype=np.float32)
    yy, xx = np.meshgrid(np.linspace(-1, 1, H), np.linspace(-1, 1, W), indexing='ij')

    circles = [
        ((-0.4, -0.4), 0.25, (1.0, 0.2, 0.2)),  # 红
        ((0.4,  0.4),  0.25, (0.2, 0.6, 1.0)),  # 蓝
        ((0.4,  -0.4), 0.2,  (0.2, 0.9, 0.3)),  # 绿
        ((-0.4, 0.4),  0.2,  (1.0, 0.8, 0.1)),  # 黄
        ((0.0,  0.0),  0.15, (0.8, 0.2, 0.8)),  # 紫
    ]

    for (cy, cx), r, color in circles:
        mask = (yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2
        img[mask] = color

    return torch.from_numpy(img)


target = make_target_image(cfg.H, cfg.W).to(device)

plt.figure(figsize=(5, 5))
plt.imshow(target.cpu().numpy())
plt.title('目标图像')
plt.axis('off')
plt.show()

## 5. 训练主循环

In [ ]:
gs_model = GaussianSplatting2D(cfg.n_gaussians, cfg.H, cfg.W).to(device)

# 不同参数用不同学习率，位置需要更慢调整以防跳出场景边界
optimizer = optim.Adam([
    {'params': gs_model.positions,  'lr': cfg.lr_pos},
    {'params': gs_model.log_scales, 'lr': cfg.lr_scale},
    {'params': gs_model.rotations,  'lr': cfg.lr_pos},
    {'params': gs_model.colors,     'lr': cfg.lr_color},
    {'params': gs_model.opacities,  'lr': cfg.lr_opacity},
])

losses = []
snapshots = []

for iteration in range(cfg.n_iters):
    pred = gs_model()
    # L1 + L2 混合损失：L2 使训练稳定，L1 保留细节
    loss = ((pred - target) ** 2).mean() + 0.1 * (pred - target).abs().mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (iteration + 1) in [1, 100, 500, 1000, 3000]:
        snapshots.append((iteration + 1, pred.detach().cpu().numpy()))

    if (iteration + 1) % 500 == 0:
        print(f'Iter {iteration+1:5d}/{cfg.n_iters}  loss={loss.item():.6f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses)
ax.set_title('3D Gaussian Splatting（2D 版）训练损失')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
plt.tight_layout()
plt.show()

## 6. 训练过程可视化

In [ ]:
fig, axes = plt.subplots(1, len(snapshots) + 1, figsize=(4 * (len(snapshots) + 1), 4))

axes[0].imshow(target.cpu().numpy())
axes[0].set_title('目标')
axes[0].axis('off')

for ax, (it, img) in zip(axes[1:], snapshots):
    ax.imshow(img.clip(0, 1))
    ax.set_title(f'Iter {it}')
    ax.axis('off')

plt.suptitle('3D Gaussian Splatting 训练进度')
plt.tight_layout()
plt.show()

## 7. 高斯点云可视化

In [ ]:
@torch.no_grad()
def visualize_gaussians(model):
    from matplotlib.patches import Ellipse

    pos  = model.positions.cpu().numpy()
    scales = torch.exp(model.log_scales).cpu().numpy()
    colors = torch.sigmoid(model.colors).cpu().numpy()
    opacs  = torch.sigmoid(model.opacities).cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # 左：高斯中心位置散点
    axes[0].scatter(pos[:, 0], pos[:, 1], c=colors, s=opacs * 100, alpha=0.8)
    axes[0].set_xlim(-1.5, 1.5)
    axes[0].set_ylim(-1.5, 1.5)
    axes[0].set_aspect('equal')
    axes[0].set_title('高斯中心位置（颜色=RGB，大小=不透明度）')

    # 右：最终渲染
    pred = model().cpu().numpy()
    axes[1].imshow(pred.clip(0, 1))
    axes[1].set_title('最终渲染结果')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    opacity_hist = opacs
    fig2, ax2 = plt.subplots(figsize=(8, 3))
    ax2.hist(opacity_hist, bins=30, color='steelblue', alpha=0.8)
    ax2.set_title('不透明度分布（低不透明度高斯应剪枝）')
    ax2.set_xlabel('Opacity')
    plt.tight_layout()
    plt.show()


visualize_gaussians(gs_model)

## 8. 密度自适应控制（Adaptive Density Control）

3DGS 的关键训练技巧，使高斯数量在训练过程中动态调整：

### 克隆（Clone）
- 触发条件：某个高斯的位置梯度很大（说明区域细节不足）
- 操作：复制该高斯，并将两者稍微错开
- 目的：在欠重建区域增加高斯密度

### 分裂（Split）
- 触发条件：某个高斯的尺度很大（一个大高斯覆盖了多个不同区域）
- 操作：将大高斯分裂为两个小高斯
- 目的：提升局部分辨率

### 剪枝（Pruning）
- 触发条件：不透明度 $\alpha$ 低于阈值（该高斯对渲染几乎无贡献）
- 操作：删除该高斯
- 目的：防止无效高斯浪费内存和计算

In [ ]:
@torch.no_grad()
def prune_gaussians(model, opacity_threshold=0.01):
    opacs = torch.sigmoid(model.opacities)
    keep  = opacs > opacity_threshold
    n_before = keep.shape[0]
    n_after  = keep.sum().item()

    model.positions.data  = model.positions.data[keep]
    model.log_scales.data = model.log_scales.data[keep]
    model.rotations.data  = model.rotations.data[keep]
    model.colors.data     = model.colors.data[keep]
    model.opacities.data  = model.opacities.data[keep]

    print(f'剪枝前：{n_before} 个高斯  →  剪枝后：{n_after} 个高斯')
    return model


gs_model = prune_gaussians(gs_model, opacity_threshold=0.05)

# 剪枝后渲染检查
with torch.no_grad():
    final = gs_model().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(target.cpu().numpy())
axes[0].set_title('目标')
axes[0].axis('off')
axes[1].imshow(final.clip(0, 1))
axes[1].set_title('剪枝后渲染')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 9. NeRF vs 3DGS 全面对比

| 维度 | NeRF（原始） | Instant-NGP | 3D Gaussian Splatting |
|------|------------|-------------|----------------------|
| 场景表达 | 隐式 MLP | 哈希编码 + 小 MLP | 显式 3D 高斯点云 |
| 训练时间 | 1-2 天 | 数分钟 | 数分钟–1 小时 |
| 渲染速度 | 慢（<1 FPS） | 中（数 FPS） | 实时（30-100+ FPS） |
| 内存（场景） | 小（网络权重 MB 级） | 中 | 大（高斯点云 GB 级） |
| 编辑性 | 差 | 差 | 好（直接操作高斯） |
| 图像质量 | 高 | 高 | 更高（细节更丰富） |

**当前趋势**：3DGS 已成为实时 3D 场景表达的主流方向，广泛应用于 AR/VR、游戏、数字人、自动驾驶场景重建等。